# Interactive Hand Mask Labeling Tool

This notebook provides an interactive OpenCV-based interface for manually drawing masks on images from your dataset.


## 1. Optional: Inspect Excel Structure

The loader scans folders automatically but filters by IDs found in Excel.
This cell is optional - it just shows the Excel structure for reference.


In [3]:
! pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]



In [ ]:
# Optional: Quick inspection of Excel file structure
# Note: The loader now scans folders automatically, so Excel is not required
# Uncomment below if you want to inspect Excel for reference:
# import pandas as pd
# df_preview = pd.read_excel("../pivot_table.xlsx")
# print("Excel columns:", list(df_preview.columns))
# print("\nFirst few rows:")
# print(df_preview.head())
# print("\nData types:")
# print(df_preview.dtypes)


Excel columns: ['id', 'angle', 'container', 'volume', 'oil', 'date', 'distance']

First few rows:
     id  angle    container volume         oil       date  distance
0  2621     60        petri  20 ml  Sunf_0_100  20_Dec_25       NaN
1  2622     60        petri  20 ml  Sunf_0_100  20_Dec_25       NaN
2  2623     60  ceramic cup  20 ml  Sunf_0_100  20_Dec_25       NaN
3  2624     60  ceramic cup  20 ml  Sunf_0_100  20_Dec_25       NaN
4  2625     60  ceramic cup  20 ml  Sunf_0_100  20_Dec_25       NaN

Data types:
id             int64
angle          int64
container     object
volume        object
oil           object
date          object
distance     float64
dtype: object


## 2. Configuration and Dataset Loading


In [32]:
import cv2
import numpy as np
import pandas as pd
import os
from pathlib import Path
from typing import List, Optional, Callable, Dict, Set

# ========== CONFIGURATION ==========

# Excel file containing valid sample IDs to filter by
EXCEL_FILE = "../pivot_table.xlsx"  # Path to your Excel file
EXCEL_ID_COLUMN = "id"  # Column name in Excel that contains sample IDs

# Dataset structure: root_folder\<capture_date>\<sample_id>\<sample_id>.png
# The loader will automatically scan all folders and find samples
# Only samples whose IDs are in the Excel file will be loaded
ROOT_IMAGE_DIR = "D:/oil/"  # Root folder for images

# Output folder for masks
OUTPUT_MASK_FOLDER = "hand_masks"

# File to track annotated sample IDs
ANNOTATED_IDS_FILE = "annotated_ids.npy"  # Numpy file to save annotated IDs

# Custom naming function for saved masks
def mask_naming_function(sample_id: str, capture_date: str = "", row_data: Optional[dict] = None) -> str:
    """
    Customize this function to define how mask files should be named.
    
    Args:
        sample_id: Sample ID
        capture_date: Capture date (optional, may be empty if not found)
        row_data: Optional dictionary with additional data
    
    Returns:
        New filename for the mask
    """
    # Example: sample_id_mask.png
    return f"{sample_id}_mask.png"
    
    # Alternative: include date in name if available
    # if capture_date:
    #     return f"{capture_date}_{sample_id}_mask.png"
    # return f"{sample_id}_mask.png"

# Output format for masks
OUTPUT_FORMAT = '.png'  # PNG recommended for masks

print("Configuration loaded!")


Configuration loaded!


## 2. Load Valid IDs from Excel and Scan Folders

The loader will:
1. Load valid sample IDs from Excel file
2. Scan folder structure to find all samples
3. Only include samples whose IDs are present in Excel


In [33]:
def load_valid_ids_from_excel(excel_path: str, id_column: str) -> Set[str]:
    """
    Load valid sample IDs from Excel file.
    
    Args:
        excel_path: Path to Excel file
        id_column: Column name containing sample IDs
    
    Returns:
        Set of valid sample IDs (as strings)
    """
    try:
        df = pd.read_excel(excel_path)
        print(f"Loaded Excel file with {len(df)} rows")
        print(f"Columns: {list(df.columns)}")
        
        if id_column not in df.columns:
            print(f"Error: Column '{id_column}' not found in Excel.")
            print(f"Available columns: {list(df.columns)}")
            return set()
        
        # Convert IDs to strings and create set
        valid_ids = set(df[id_column].astype(str).str.strip())
        print(f"Found {len(valid_ids)} unique IDs in Excel column '{id_column}'")
        
        return valid_ids
        
    except Exception as e:
        print(f"Error loading Excel file: {e}")
        import traceback
        traceback.print_exc()
        return set()

def load_dataset_from_folders(root_dir: str, valid_ids: Set[str]) -> List[dict]:
    """
    Load image paths by scanning folder structure:
    root_folder\<capture_date>\<sample_id>\<sample_id>.png
    
    This function is date-agnostic - it automatically discovers all samples
    by drilling through the folder structure. Only samples whose IDs are in
    valid_ids will be included.
    
    Args:
        root_dir: Root directory for images
        valid_ids: Set of valid sample IDs (from Excel) to filter by
    
    Returns:
        List of dictionaries, each containing image info and path
    """
    dataset = []
    root_path = Path(root_dir)
    
    if not root_path.exists():
        print(f"Error: Root directory does not exist: {root_dir}")
        return []
    
    print(f"Scanning folder structure: {root_dir}")
    print("Looking for structure: root_folder/<date>/<sample_id>/<sample_id>.png")
    print(f"Filtering by {len(valid_ids)} valid IDs from Excel")
    
    found_count = 0
    skipped_count = 0
    
    # Walk through the directory structure
    # Level 1: date folders (or any folder name)
    for date_folder in root_path.iterdir():
        if not date_folder.is_dir():
            continue
        
        capture_date = date_folder.name
        print(f"  Found date folder: {capture_date}")
        
        # Level 2: sample_id folders
        for sample_folder in date_folder.iterdir():
            if not sample_folder.is_dir():
                continue
            
            sample_id = sample_folder.name
            
            # Check if this sample_id is in the valid IDs from Excel
            if sample_id not in valid_ids:
                skipped_count += 1
                continue  # Skip this sample - not in Excel
            
            # Look for <sample_id>.png in this folder
            expected_image = sample_folder / f"{sample_id}.png"
            
            if expected_image.exists():
                dataset.append({
                    'index': len(dataset),
                    'path': str(expected_image),
                    'sample_id': sample_id,
                    'capture_date': capture_date,
                    'row_data': {}  # No Excel data, but keep structure consistent
                })
                found_count += 1
                print(f"    ✓ Found sample: {sample_id} (date: {capture_date})")
            else:
                # Try to find any .png file in the folder
                png_files = list(sample_folder.glob("*.png"))
                if png_files:
                    # Use the first PNG found
                    img_file = png_files[0]
                    dataset.append({
                        'index': len(dataset),
                        'path': str(img_file),
                        'sample_id': sample_id,
                        'capture_date': capture_date,
                        'row_data': {}
                    })
                    found_count += 1
                    print(f"    ✓ Found sample: {sample_id} (date: {capture_date}) - using {img_file.name}")
                else:
                    print(f"    ✗ No image found for sample: {sample_id} (in Excel but no image)")
    
    print(f"\nSuccessfully loaded {found_count} samples from folder structure")
    if skipped_count > 0:
        print(f"Skipped {skipped_count} samples not found in Excel")
    return dataset

# Step 1: Load valid IDs from Excel
print("=" * 50)
print("Step 1: Loading valid IDs from Excel...")
print("=" * 50)
valid_ids = load_valid_ids_from_excel(EXCEL_FILE, EXCEL_ID_COLUMN)

if len(valid_ids) == 0:
    print("\nERROR: No valid IDs found in Excel. Please check:")
    print("1. Excel file path is correct")
    print("2. Column name matches your Excel structure")
else:
    print(f"\nValid IDs loaded: {len(valid_ids)}")
    print(f"Sample IDs (first 10): {sorted(list(valid_ids))[:10]}")

# Step 2: Load dataset from folders (filtered by Excel IDs)
print("\n" + "=" * 50)
print("Step 2: Scanning folders and filtering by Excel IDs...")
print("=" * 50)
dataset = load_dataset_from_folders(ROOT_IMAGE_DIR, valid_ids)

if len(dataset) == 0:
    print("\nWARNING: No images found in dataset. Please check:")
    print("1. ROOT_IMAGE_DIR is set correctly")
    print("2. Folder structure matches: root/<date>/<sample_id>/<sample_id>.png")
    print("3. Sample IDs in folders match IDs in Excel")
    print("4. Images exist in the expected locations")
else:
    print(f"\nDataset ready: {len(dataset)} images loaded")
    print(f"Sample IDs (first 10): {[d['sample_id'] for d in dataset[:10]]}")
    print(f"Date folders found: {sorted(set(d['capture_date'] for d in dataset))}")
    
    # Check if there are Excel IDs without corresponding images
    loaded_ids = set(d['sample_id'] for d in dataset)
    missing_ids = valid_ids - loaded_ids
    if missing_ids:
        print(f"\nNote: {len(missing_ids)} IDs from Excel have no corresponding images:")
        print(f"Missing IDs (first 10): {sorted(list(missing_ids))[:10]}")


Step 1: Loading valid IDs from Excel...
Loaded Excel file with 334 rows
Columns: ['id', 'angle', 'container', 'volume', 'oil', 'date', 'distance']
Found 333 unique IDs in Excel column 'id'

Valid IDs loaded: 333
Sample IDs (first 10): ['2621', '2622', '2623', '2624', '2625', '2627', '2628', '2629', '2630', '2631']

Step 2: Scanning folders and filtering by Excel IDs...
Scanning folder structure: D:/oil/
Looking for structure: root_folder/<date>/<sample_id>/<sample_id>.png
Filtering by 333 valid IDs from Excel
  Found date folder: 17_Dec_2025
  Found date folder: 20_Dec_2025
    ✓ Found sample: 2621 (date: 20_Dec_2025)
    ✓ Found sample: 2622 (date: 20_Dec_2025)
    ✓ Found sample: 2623 (date: 20_Dec_2025)
    ✓ Found sample: 2624 (date: 20_Dec_2025)
    ✓ Found sample: 2625 (date: 20_Dec_2025)
    ✓ Found sample: 2627 (date: 20_Dec_2025)
  Found date folder: 21_Dec_2025
    ✓ Found sample: 2628 (date: 21_Dec_2025)
    ✓ Found sample: 2629 (date: 21_Dec_2025)
    ✓ Found sample: 2630 (

## 3. Interactive Labeling Interface


In [34]:
# Load previously annotated IDs if file exists
def load_annotated_ids(file_path: str) -> set:
    """Load set of annotated sample IDs from numpy file."""
    if os.path.exists(file_path):
        try:
            ids = np.load(file_path, allow_pickle=True)
            if isinstance(ids, np.ndarray):
                ids = ids.tolist()
            return set(ids)
        except:
            return set()
    return set()

def save_annotated_ids(file_path: str, annotated_ids: set):
    """Save set of annotated sample IDs to numpy file."""
    np.save(file_path, np.array(list(annotated_ids)))
    print(f"Saved {len(annotated_ids)} annotated IDs to {file_path}")

# Initialize annotated IDs
annotated_ids = load_annotated_ids(ANNOTATED_IDS_FILE)
print(f"Loaded {len(annotated_ids)} previously annotated sample IDs")


Loaded 4 previously annotated sample IDs


## 4. Interactive Labeling Interface Class


In [35]:
class HandLabelingTool:
    """
    Interactive OpenCV-based tool for manually drawing masks on images.
    
    Controls:
    - Left mouse button: Draw mask (hold and drag)
    - Right mouse button: Erase mask (hold and drag)
    - 's' key: Save current mask and move to next image
    - 'n' key: Skip current image (move to next without saving)
    - 'r' key: Reset current mask
    - 'q' or ESC: Quit labeling session
    - Arrow keys: Navigate between images
    """
    
    def __init__(self, dataset: List[dict], output_folder: str, naming_function: Callable):
        self.dataset = dataset
        self.output_folder = Path(output_folder)
        self.output_folder.mkdir(parents=True, exist_ok=True)
        self.naming_function = naming_function
        
        self.current_idx = 0
        self.drawing = False
        self.mode = 'draw'  # 'draw' or 'erase'
        self.brush_size = 10
        
        # Mouse position tracking for brush indicator
        self.mouse_x = -1
        self.mouse_y = -1
        
        # Load image and initialize mask
        self.load_current_image()
        
    def load_current_image(self):
        """Load the current image and initialize mask."""
        if self.current_idx >= len(self.dataset):
            print("All images processed!")
            return False
        
        item = self.dataset[self.current_idx]
        self.current_image_path = item['path']
        self.current_sample_id = item['sample_id']
        self.current_capture_date = item['capture_date']
        self.current_row_data = item.get('row_data', {})
        
        # Load image
        self.original_image = cv2.imread(self.current_image_path)
        if self.original_image is None:
            print(f"Error: Could not load image {self.current_image_path}")
            return False
        
        # Initialize mask (black background, white for mask)
        self.mask = np.zeros((self.original_image.shape[0], self.original_image.shape[1]), dtype=np.uint8)
        
        # Reset mouse position for new image
        self.mouse_x = -1
        self.mouse_y = -1
        
        # Create display image (original + mask overlay)
        self.update_display()
        
        print(f"\\nImage {self.current_idx + 1}/{len(self.dataset)}")
        print(f"Sample ID: {self.current_sample_id}")
        print(f"Capture Date: {self.current_capture_date}")
        print(f"Path: {self.current_image_path}")
        
        return True
    
    def update_display(self):
        """Update the display image with mask overlay."""
        # Create overlay: green/red tint for masked areas
        overlay = self.original_image.copy()
        
        # Create colored mask (green and red channels for visibility)
        mask_colored = np.zeros_like(self.original_image)
        mask_colored[:, :, 1] = self.mask  # Green channel
        mask_colored[:, :, 2] = self.mask  # Red channel (for visibility)
        
        # Blend overlay where mask is non-zero
        mask_bool = self.mask > 0
        if np.any(mask_bool):
            # Use numpy operations for blending (more reliable than cv2.addWeighted with indexing)
            overlay[mask_bool] = (
                overlay[mask_bool].astype(np.float32) * 0.6 + 
                mask_colored[mask_bool].astype(np.float32) * 0.4
            ).astype(np.uint8)
        
        # Draw red brush indicator at mouse position (if mouse is over image)
        if self.mouse_x >= 0 and self.mouse_y >= 0:
            # Check if mouse is within image bounds
            if (0 <= self.mouse_x < overlay.shape[1] and 
                0 <= self.mouse_y < overlay.shape[0]):
                # Draw red brush circle with thin edge
                # Create a temporary overlay for the brush indicator
                brush_overlay = overlay.copy()
                # Inner red circle (semi-transparent fill)
                cv2.circle(brush_overlay, (self.mouse_x, self.mouse_y), self.brush_size, (0, 0, 255), -1)
                # Blend with alpha (keep same transparency)
                alpha = 0.3
                overlay = cv2.addWeighted(overlay, 1 - alpha, brush_overlay, alpha, 0)
                # Red edge (thin pixel lining on the edge)
                cv2.circle(overlay, (self.mouse_x, self.mouse_y), self.brush_size, (0, 0, 255), 1)
        
        # Add mode indicator (black text with white outline for visibility)
        mode_text = f"Mode: {'DRAW' if self.mode == 'draw' else 'ERASE'} | Brush: {self.brush_size} | +/- to adjust"
        # Draw white outline first for visibility
        cv2.putText(overlay, mode_text, (10, overlay.shape[0] - 20), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 4)
        # Then draw black text on top
        cv2.putText(overlay, mode_text, (10, overlay.shape[0] - 20), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
        
        # Add instructions (black text with white outline)
        instructions = [
            "Left Click: Draw | Right Click: Erase | 's': Save | 'n': Next | 'r': Reset | 'q': Quit",
            f"Sample: {self.current_sample_id} | [{self.current_idx + 1}/{len(self.dataset)}]"
        ]
        for i, text in enumerate(instructions):
            # Draw white outline first
            cv2.putText(overlay, text, (10, overlay.shape[0] - 60 + i * 20), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 3)
            # Then draw black text on top
            cv2.putText(overlay, text, (10, overlay.shape[0] - 60 + i * 20), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
        
        self.display_image = overlay
    
    def mouse_callback(self, event, x, y, flags, param):
        """Handle mouse events for drawing and brush size adjustment."""
        # Always track mouse position for brush indicator
        self.mouse_x = x
        self.mouse_y = y
        
        if event == cv2.EVENT_LBUTTONDOWN:
            self.drawing = True
            self.mode = 'draw'
            # Apply drawing immediately on click
            cv2.circle(self.mask, (x, y), self.brush_size, 255, -1)
            self.update_display()
        elif event == cv2.EVENT_RBUTTONDOWN:
            self.drawing = True
            self.mode = 'erase'
            # Apply erasing immediately on click
            cv2.circle(self.mask, (x, y), self.brush_size, 0, -1)
            self.update_display()
        elif event == cv2.EVENT_MOUSEMOVE:
            # Apply drawing/erasing only when button is pressed
            if self.drawing:
                if self.mode == 'draw':
                    cv2.circle(self.mask, (x, y), self.brush_size, 255, -1)
                else:  # erase
                    cv2.circle(self.mask, (x, y), self.brush_size, 0, -1)
            # Display update happens in main loop for smooth brush following
        elif event == cv2.EVENT_LBUTTONUP or event == cv2.EVENT_RBUTTONUP:
            self.drawing = False
            # Display update happens in main loop
        elif event == cv2.EVENT_MOUSEWHEEL:
            # Adjust brush size with mouse wheel
            # On Windows: flags > 0 means scroll up, flags < 0 means scroll down
            # The value is typically 120 or -120, but we just check sign
            delta = flags
            if delta > 0:  # Scroll up - increase brush size
                self.brush_size = min(50, self.brush_size + 2)
            elif delta < 0:  # Scroll down - decrease brush size
                self.brush_size = max(1, self.brush_size - 2)
            self.update_display()
    
    def save_mask(self) -> bool:
        """Save the current mask."""
        # Generate mask filename
        mask_filename = self.naming_function(
            self.current_sample_id, 
            self.current_capture_date,
            self.current_row_data
        )
        
        # Ensure correct extension
        if not mask_filename.endswith(OUTPUT_FORMAT):
            name, _ = os.path.splitext(mask_filename)
            mask_filename = f"{name}{OUTPUT_FORMAT}"
        
        # Save mask
        output_path = self.output_folder / mask_filename
        cv2.imwrite(str(output_path), self.mask)
        
        print(f"Mask saved: {output_path}")
        return True
    
    def reset_mask(self):
        """Reset the current mask."""
        self.mask = np.zeros_like(self.mask)
        self.update_display()
        print("Mask reset")
    
    def next_image(self, save_current: bool = False):
        """Move to the next image."""
        if save_current:
            self.save_mask()
        
        self.current_idx += 1
        if self.current_idx < len(self.dataset):
            return self.load_current_image()
        else:
            print("\\nAll images processed!")
            return False
    
    def previous_image(self):
        """Move to the previous image."""
        if self.current_idx > 0:
            self.current_idx -= 1
            return self.load_current_image()
        return False
    
    def run(self, annotated_ids: set):
        """Run the interactive labeling session."""
        window_name = "Hand Labeling Tool - Press 'q' to quit"
        cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
        
        # Create wrapper for mouse callback to access instance
        def mouse_wrapper(event, x, y, flags, param):
            self.mouse_callback(event, x, y, flags, param)
        
        cv2.setMouseCallback(window_name, mouse_wrapper)
        
        print("\\n=== Labeling Tool Started ===")
        print("Controls:")
        print("  Left Click + Drag: Draw mask")
        print("  Right Click + Drag: Erase mask")
        print("  Mouse Wheel Up/Down: Increase/Decrease brush size")
        print("  +/- keys: Adjust brush size")
        print("  's': Save mask and go to next image")
        print("  'n': Skip to next image (don't save)")
        print("  'r': Reset current mask")
        print("  Arrow Right/Left: Navigate images")
        print("  'q' or ESC: Quit")
        print("=" * 30)
        
        while True:
            # Continuously update display to show brush indicator following cursor
            self.update_display()
            cv2.imshow(window_name, self.display_image)
            key = cv2.waitKey(1) & 0xFF
            
            if key == ord('q') or key == 27:  # 'q' or ESC
                print("\\nQuitting labeling session...")
                break
            elif key == ord('s'):  # Save and next
                self.save_mask()
                annotated_ids.add(self.current_sample_id)
                if not self.next_image():
                    break
            elif key == ord('n'):  # Next without saving
                if not self.next_image():
                    break
            elif key == ord('r'):  # Reset mask
                self.reset_mask()
            elif key == ord('+') or key == ord('='):  # Increase brush size
                self.brush_size = min(50, self.brush_size + 2)
                self.update_display()
            elif key == ord('-') or key == ord('_'):  # Decrease brush size
                self.brush_size = max(1, self.brush_size - 2)
                self.update_display()
            elif key == 82 or key == 83:  # Arrow keys (may vary by system)
                if key == 82:  # Up
                    self.brush_size = min(50, self.brush_size + 2)
                    self.update_display()
                elif key == 84:  # Down
                    self.brush_size = max(1, self.brush_size - 2)
                    self.update_display()
            elif key == 81 or key == 2:  # Left arrow
                self.previous_image()
            elif key == 83 or key == 3:  # Right arrow
                pass  # Already handled by 'n' or 's'
        
        cv2.destroyAllWindows()
        return annotated_ids

print("HandLabelingTool class defined!")


HandLabelingTool class defined!


In [25]:
cv2.imread(dataset[0]['path'])

array([[[232, 235, 235],
        [243, 245, 242],
        [226, 228, 230],
        ...,
        [227, 226, 225],
        [238, 239, 238],
        [238, 234, 237]],

       [[232, 233, 236],
        [246, 246, 243],
        [227, 229, 232],
        ...,
        [221, 220, 224],
        [239, 235, 235],
        [229, 235, 234]],

       [[232, 230, 236],
        [241, 245, 244],
        [226, 228, 230],
        ...,
        [229, 225, 223],
        [241, 238, 236],
        [238, 232, 234]],

       ...,

       [[198, 210, 216],
        [214, 221, 223],
        [197, 203, 214],
        ...,
        [204, 210, 214],
        [222, 224, 224],
        [214, 220, 223]],

       [[200, 208, 219],
        [210, 219, 226],
        [202, 207, 217],
        ...,
        [205, 209, 216],
        [222, 222, 226],
        [215, 219, 225]],

       [[200, 206, 214],
        [210, 220, 224],
        [201, 208, 215],
        ...,
        [205, 209, 212],
        [215, 223, 225],
        [217, 220, 223]]

In [39]:
# Filter dataset to only unannotated images (optional - comment out to label all)
# Uncomment the following lines to skip already annotated images:
unannotated_dataset = [d for d in dataset if d['sample_id'] not in annotated_ids]
print(f"Filtered: {len(dataset)} total, {len(unannotated_dataset)} unannotated")
#dataset = unannotated_dataset

# Create and run labeling tool
if len(dataset) > 0:
    tool = HandLabelingTool(
        dataset=dataset,
        output_folder=OUTPUT_MASK_FOLDER,
        naming_function=mask_naming_function
    )
    
    # Run the labeling session
    annotated_ids = tool.run(annotated_ids)
    
    # Save annotated IDs
    save_annotated_ids(ANNOTATED_IDS_FILE, annotated_ids)
    
    print(f"\\n=== Session Complete ===")
    print(f"Total annotated: {len(annotated_ids)}")
    print(f"Remaining: {len(dataset) - len(annotated_ids)}")
else:
    print("No images to label!")


Filtered: 210 total, 0 unannotated
\nImage 1/210
Sample ID: 2625
Capture Date: 20_Dec_2025
Path: D:\oil\20_Dec_2025\2625\2625.png
\n=== Labeling Tool Started ===
Controls:
  Left Click + Drag: Draw mask
  Right Click + Drag: Erase mask
  Mouse Wheel Up/Down: Increase/Decrease brush size
  +/- keys: Adjust brush size
  's': Save mask and go to next image
  'n': Skip to next image (don't save)
  'r': Reset current mask
  Arrow Right/Left: Navigate images
  'q' or ESC: Quit
Mask saved: hand_masks\2625_mask.png
\nImage 2/210
Sample ID: 2627
Capture Date: 20_Dec_2025
Path: D:\oil\20_Dec_2025\2627\2627.png
Mask saved: hand_masks\2627_mask.png
\nImage 3/210
Sample ID: 2628
Capture Date: 21_Dec_2025
Path: D:\oil\21_Dec_2025\2628\2628.png
Mask saved: hand_masks\2628_mask.png
\nImage 4/210
Sample ID: 2629
Capture Date: 21_Dec_2025
Path: D:\oil\21_Dec_2025\2629\2629.png
\nQuitting labeling session...
Saved 214 annotated IDs to annotated_ids.npy
\n=== Session Complete ===
Total annotated: 214
Rema

## 6. Utility Functions

### View Annotated IDs


In [38]:
# View currently annotated sample IDs
print("Annotated Sample IDs:")
annotated_list = sorted(list(annotated_ids))
print(annotated_list)
print(f"\\nTotal: {len(annotated_list)}")


Annotated Sample IDs:
['2621', '2622', '2623', '2624', '2625', '2627', '2628', '2629', '2630', '2631', '2632', '2633', '2634', '2635', '2636', '2642', '2644', '2645', '2646', '2647', '2649', '2650', '2651', '2652', '2653', '2654', '2656', '2657', '2658', '2659', '2662', '2663', '2664', '2665', '2666', '2667', '2668', '2669', '2672', '2673', '2674', '2675', '2676', '2677', '2678', '2679', '2680', '2681', '2682', '2683', '2685', '2687', '2688', '2689', '2690', '2691', '2692', '2693', '2694', '2695', '2696', '2697', '2698', '2699', '2700', '2701', '2702', '2703', '2704', '2705', '2706', '2707', '2708', '2709', '2710', '2711', '2712', '2713', '2714', '2716', '2722', '2724', '2725', '2726', '2727', '2728', '2729', '2730', '2731', '2732', '2733', '2734', '2735', '2736', '2737', '2738', '2739', '2740', '2741', '2742', '2743', '2744', '2745', '2746', '2747', '2748', '2749', '2750', '2751', '2752', '2753', '2754', '2755', '2756', '2757', '2758', '2759', '2760', '2761', '2762', '2765', '2766', '

### Filter Dataset to Unannotated Only


In [ ]:
# Get list of unannotated sample IDs
unannotated = [d for d in dataset if d['sample_id'] not in annotated_ids]
print(f"Unannotated images: {len(unannotated)}")
if len(unannotated) > 0:
    print(f"Sample IDs: {[d['sample_id'] for d in unannotated[:10]]}...")


### Resume Labeling from Specific Index


In [ ]:
# To resume labeling from a specific image index, modify the dataset:
# For example, to start from image 10:
# START_INDEX = 10
# dataset_subset = dataset[START_INDEX:]
# tool = HandLabelingTool(dataset_subset, OUTPUT_MASK_FOLDER, mask_naming_function)
# annotated_ids = tool.run(annotated_ids)
# save_annotated_ids(ANNOTATED_IDS_FILE, annotated_ids)
